# PUDL EIA-923 data maturity comparison: incremental_ytd vs. provisional vs. final (2024)

EIA-923 data for a given year passes through several PUDL "data maturity" stages before OGE can use "final" data
in a release: `incremental_ytd` (monthly-cadence updates during the year), `provisional` (EIA's summer early
release), and finally `final` (published ~a year after the data year). PUDL only retains the *latest* maturity for
each row, so comparing all three stages for the same data year requires reading three different **pinned PUDL
release versions**, each of which happened to hold 2024 data at a different maturity when it was built:

| snapshot label | PUDL version | release date | expected 2024 `data_maturity` |
|---|---|---|---|
| `incremental_ytd` | `v2025.2.0` | 2025-02-13 | `incremental_ytd` / `monthly_update` |
| `provisional` | `v2025.7.0` | 2025-07-03 | `provisional` |
| `final` | `v2025.11.0` | 2025-11-13 | `final` |

This notebook focuses on the three EIA-923 monthly tables OGE relies on most directly, and on the two columns that
matter most for emissions calculations, `fuel_consumed_mmbtu` and `net_generation_mwh` (`out_eia923__monthly_boiler_fuel`
only has the former, `out_eia923__monthly_generation` only has the latter). Each table is read directly from
`s3://pudl.catalyst.coop/{version}/` for each pinned version (anonymous S3 access, no download of the full
database), and compared for:

1. **Completeness** — which plant-months are present in the early snapshots vs. missing until final.
2. **Accuracy** — for rows present in both an early snapshot and final, how much do the reported values change.

Caveat: PUDL's `v2025.11.0` release notes mention a bug fix to how `data_maturity` was labeled for EIA-923 in
*earlier* versions. This notebook treats each snapshot's actual `data_maturity` values as something to observe and
report (Section 1), not something to assume from the version/label above.

No data is cached locally — every run re-reads the three pinned versions directly from S3.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

%reload_ext autoreload
%autoreload 2

import sys

sys.path.append("../../src")

from oge.column_checks import apply_dtypes

YEAR = 2024

# snapshot label -> pinned PUDL version holding YEAR data at that maturity
VERSIONS = {
    "incremental_ytd": "v2025.2.0",
    "provisional": "v2025.7.0",
    "final": "v2025.11.0",
}

## Section 1: Load the three PUDL snapshots

In [ ]:
def read_pudl_table_at_version(table_name, version, year, columns=None):
    """Read one PUDL table for one calendar year from a pinned PUDL release version on S3.

    Args:
        table_name (str): PUDL table name, e.g. "out_eia923__monthly_generation".
        version (str): pinned PUDL release tag, e.g. "v2025.11.0".
        year (int): calendar year to filter `report_date` to.
        columns (list[str]): columns to read; None reads all columns.

    Returns:
        pd.DataFrame: the filtered table, with OGE dtypes applied where registered.
    """
    path = f"s3://pudl.catalyst.coop/{version}/{table_name}.parquet"
    filters = [
        ("report_date", ">=", pd.Timestamp(year, 1, 1)),
        ("report_date", "<=", pd.Timestamp(year, 12, 1)),
    ]
    df = pd.read_parquet(
        path, filters=filters, columns=columns, storage_options={"anon": True}
    )
    return apply_dtypes(df)

In [ ]:
# key_cols: columns that uniquely identify a row within a single year of this table
# numeric_cols: value columns to compare for accuracy (row-level ratio + national total) --
# restricted to fuel_consumed_mmbtu/net_generation_mwh, whichever the table actually has
# monthly: whether report_date varies within the year (True) or is fixed at Jan 1 (False)
TABLES = {
    "out_eia923__monthly_generation_fuel_combined": {
        "key_cols": [
            "plant_id_eia",
            "report_date",
            "prime_mover_code",
            "energy_source_code",
        ],
        "read_cols": [
            "plant_id_eia",
            "report_date",
            "data_maturity",
            "prime_mover_code",
            "energy_source_code",
            "fuel_consumed_mmbtu",
            "net_generation_mwh",
        ],
        "numeric_cols": ["fuel_consumed_mmbtu", "net_generation_mwh"],
        "monthly": True,
    },
    "out_eia923__monthly_boiler_fuel": {
        "key_cols": [
            "plant_id_eia",
            "boiler_id",
            "report_date",
            "energy_source_code",
            "prime_mover_code",
        ],
        "read_cols": [
            "plant_id_eia",
            "boiler_id",
            "report_date",
            "data_maturity",
            "energy_source_code",
            "prime_mover_code",
            "fuel_consumed_mmbtu",
        ],
        "numeric_cols": ["fuel_consumed_mmbtu"],
        "monthly": True,
    },
    "out_eia923__monthly_generation": {
        "key_cols": ["plant_id_eia", "generator_id", "report_date"],
        "read_cols": [
            "plant_id_eia",
            "generator_id",
            "report_date",
            "data_maturity",
            "net_generation_mwh",
        ],
        "numeric_cols": ["net_generation_mwh"],
        "monthly": True,
    },
}

In [ ]:
# load all 3 tables x 3 snapshots directly from S3 (9 reads, no local caching)
snapshots = {}
for table_name, spec in TABLES.items():
    for label, version in VERSIONS.items():
        snapshots[(table_name, label)] = read_pudl_table_at_version(
            table_name, version, YEAR, columns=spec["read_cols"]
        )

In [ ]:
# row counts and observed data_maturity distribution per (table, snapshot) -- see Section 1 caveat above:
# don't assume a snapshot is purely one maturity value, check it
load_summary = []
for (table_name, label), df in snapshots.items():
    maturity_counts = df["data_maturity"].value_counts(dropna=False).to_dict()
    load_summary.append(
        {
            "table": table_name,
            "snapshot": label,
            "version": VERSIONS[label],
            "n_rows": len(df),
            "data_maturity_values": maturity_counts,
        }
    )
load_summary = pd.DataFrame(load_summary)
load_summary

### Aside: why `v2025.2.0` and not a later "incremental" version like `v2025.5.0`?

`out_eia923__monthly_generation_fuel_combined` has zero rows for `report_date = 2024-12-01` in the `incremental_ytd`
snapshot above (`v2025.2.0`) -- EIA's monthly-reporting lag means December hadn't been submitted yet when that
version was built in Feb 2025. The next available PUDL version, `v2025.5.0` (2025-05-20), does have December rows,
so it's tempting to swap to it. But checking it directly shows why that would make things worse, not better:

In [ ]:
# quick, standalone check -- NOT added to `snapshots`/`TABLES`, since the conclusion below is that this
# version's data_maturity label can't be trusted for this kind of comparison
mislabeling_check = []
for table_name in TABLES:
    df = read_pudl_table_at_version(
        table_name, "v2025.5.0", YEAR, columns=["report_date", "data_maturity"]
    )
    final_n_rows = len(snapshots[(table_name, "final")])
    mislabeling_check.append(
        {
            "table": table_name,
            "n_rows": len(df),
            "pct_of_final_row_count": round(len(df) / final_n_rows * 100, 1),
            "data_maturity_values": df["data_maturity"]
            .value_counts(dropna=False)
            .to_dict(),
        }
    )
pd.DataFrame(mislabeling_check)

At `v2025.5.0`, all three tables are **100% labeled `final`** -- despite `out_eia923__monthly_generation_fuel_combined`
holding only ~32% of the row count `final` eventually has (and the other two tables ~76-77%). A version that's
missing two-thirds of a table's rows is not "final" by any reasonable definition. This is almost certainly the
same EIA-923 `data_maturity` labeling bug that PUDL's `v2025.11.0` release notes mention fixing, and it means
`v2025.5.0`'s `data_maturity` column can't be trusted at face value. Switching our `incremental_ytd` pin to it
would trade a real, honestly-labeled completeness gap (`v2025.2.0`, December simply absent from one table) for a
mislabeled one, without meaningfully closing that gap (row counts barely grow between the two versions). We keep
`v2025.2.0` as the `incremental_ytd` pin.

## Section 2: Completeness

For each table, compare the set of row-keys present in the `incremental_ytd`/`provisional` snapshots against the
`final` snapshot's set of keys. For the monthly EIA-923 tables this is broken out by `report_date`, since EIA notes
that monthly-cadence data only reflects generators/plants that report monthly -- annual-only reporters should show
up as a completeness gap that's worst early in the year and closes as `final` data arrives.

In [ ]:
def completeness_vs_final(table_name, by_month=False):
    """Compare key-column presence in the incremental/provisional snapshots vs. final.

    Args:
        table_name (str): key into TABLES/snapshots.
        by_month (bool): if True, group results by report_date (for monthly-cadence tables).

    Returns:
        pd.DataFrame: one row per (snapshot, [report_date]) with n_final, n_present, pct_present,
            and n_extra_in_snapshot (keys present in the snapshot but not in final).
    """
    key_cols = TABLES[table_name]["key_cols"]
    final_keys = snapshots[(table_name, "final")][key_cols].drop_duplicates()
    group_cols = ["report_date"] if by_month else []

    rows = []
    for label in ["incremental_ytd", "provisional"]:
        snap_keys = snapshots[(table_name, label)][key_cols].drop_duplicates()

        presence = final_keys.merge(
            snap_keys, on=key_cols, how="left", indicator=True, validate="1:1"
        )
        presence["present"] = presence["_merge"] == "both"

        extra = snap_keys.merge(
            final_keys, on=key_cols, how="left", indicator=True, validate="1:1"
        )
        n_extra = (extra["_merge"] == "left_only").sum()

        if group_cols:
            summary = presence.groupby(group_cols, dropna=False)["present"].agg(
                n_final="count", n_present="sum"
            )
            summary["pct_present"] = (
                summary["n_present"] / summary["n_final"] * 100
            ).round(1)
            summary = summary.reset_index()
        else:
            summary = pd.DataFrame(
                [
                    {
                        "n_final": len(presence),
                        "n_present": presence["present"].sum(),
                        "pct_present": round(presence["present"].mean() * 100, 1),
                    }
                ]
            )
        summary["snapshot"] = label
        summary["n_extra_in_snapshot"] = n_extra
        rows.append(summary)

    return pd.concat(rows, ignore_index=True)

In [ ]:
completeness_results = {
    table_name: completeness_vs_final(table_name, by_month=spec["monthly"])
    for table_name, spec in TABLES.items()
}

for table_name, result in completeness_results.items():
    print(table_name)
    display(result)

In [ ]:
# visualize the ramp described in EIA's caveat: monthly-cadence completeness should improve month over month
# as more annual-only reporters land in the *final* snapshot's denominator but stay absent from the earlier ones
monthly_table = "out_eia923__monthly_generation"
fig = px.line(
    completeness_results[monthly_table],
    x="report_date",
    y="pct_present",
    color="snapshot",
    markers=True,
    title=f"{monthly_table}: % of final plant-generator-months present, by report month",
)
fig.update_yaxes(range=[0, 105])
fig

### Characterizing what's missing

Join the missing plant-month keys back to `final`'s own attributes to see whether absence skews toward a
particular fuel type or prime mover.

In [ ]:
def missing_by_attribute(table_name, attribute_col, snapshot_label="incremental_ytd"):
    """Break down which `final`-only keys are missing from `snapshot_label`, by a final-table attribute.

    Args:
        table_name (str): key into TABLES/snapshots.
        attribute_col (str): a categorical column from the final snapshot to group by.
        snapshot_label (str): "incremental_ytd" or "provisional".

    Returns:
        pd.DataFrame: count of final keys and count missing from the snapshot, per attribute value.
    """
    key_cols = TABLES[table_name]["key_cols"]
    final_df = snapshots[(table_name, "final")]
    snap_keys = snapshots[(table_name, snapshot_label)][key_cols].drop_duplicates()

    presence = final_df.merge(
        snap_keys, on=key_cols, how="left", indicator=True, validate="m:1"
    )
    presence["missing"] = presence["_merge"] == "left_only"

    return (
        presence.groupby(attribute_col, dropna=False)["missing"]
        .agg(n_final="count", n_missing="sum")
        .assign(pct_missing=lambda d: (d["n_missing"] / d["n_final"] * 100).round(1))
    )

In [ ]:
missing_by_attribute(
    "out_eia923__monthly_generation_fuel_combined",
    "energy_source_code",
    "provisional",
)

## Section 3: Accuracy

For row-keys present in *both* an early snapshot and `final`, quantify how much the reported values changed. This
mirrors the ratio + bucket pattern `oge.validation.compare_plant_level_results_to_egrid` uses for eGRID comparisons:
`snapshot_value / final_value`, bucketed into named ranges.

Two of the bucket labels below look similar but mean different things:
- `missing_in_<snapshot>` / `missing_in_final`: the row-key itself is absent from that snapshot (Section 2's
  completeness gap).
- `missing`: the row-key is present in *both* snapshots, but the value being compared (`fuel_consumed_mmbtu` or
  `net_generation_mwh`) is null on at least one side -- i.e. the plant-month is already being reported, just not
  with a value for this particular column yet.

In [ ]:
RATIO_BINS = [
    -999999999,
    -0.0001,
    0.5,
    0.9,
    0.99,
    0.9999,
    1,
    1.0001,
    1.01,
    1.1,
    1.5,
    999999999,
]
RATIO_LABELS = [
    "negative",
    "<50%",
    "-50% to -10%",
    "-10% to -1%",
    "+/-1%",
    "!exact",
    "!exact",
    "+/-1%",
    "+1% to 10%",
    "+10% to 50%",
    ">50%",
]


def bucket_ratio(ratio):
    """Bucket a snapshot/final ratio series into named accuracy ranges (see validation.py precedent)."""
    return pd.cut(ratio, bins=RATIO_BINS, labels=RATIO_LABELS, ordered=False)


def compare_snapshot_to_final(table_name, snapshot_label):
    """Row-level snapshot-vs-final ratio comparison for one table's numeric_cols.

    Args:
        table_name (str): key into TABLES/snapshots.
        snapshot_label (str): "incremental_ytd" or "provisional".

    Returns:
        pd.DataFrame: count of row-keys per accuracy bucket, one column per numeric column.
    """
    key_cols = TABLES[table_name]["key_cols"]
    numeric_cols = TABLES[table_name]["numeric_cols"]
    if not numeric_cols:
        return pd.DataFrame()

    snap_df = snapshots[(table_name, snapshot_label)][
        key_cols + numeric_cols
    ].drop_duplicates(key_cols)
    final_df = snapshots[(table_name, "final")][
        key_cols + numeric_cols
    ].drop_duplicates(key_cols)
    merged = snap_df.merge(
        final_df,
        on=key_cols,
        how="outer",
        suffixes=("_snapshot", "_final"),
        validate="1:1",
        indicator=True,
    )

    bucket_counts = {}
    for col in numeric_cols:
        snap_col, final_col = f"{col}_snapshot", f"{col}_final"
        ratio = merged[snap_col] / merged[final_col]
        both_zero = (merged[snap_col] == 0) & (merged[final_col] == 0)
        ratio = ratio.mask(both_zero, 1)

        status = bucket_ratio(ratio).astype(str).replace("nan", "missing")
        status = status.mask(merged["_merge"] == "left_only", "missing_in_final")
        status = status.mask(
            merged["_merge"] == "right_only", f"missing_in_{snapshot_label}"
        )

        bucket_counts[col] = status.value_counts()

    return pd.DataFrame(bucket_counts).fillna(0).astype(int)

In [ ]:
accuracy_results = {}
for table_name, spec in TABLES.items():
    for label in ["incremental_ytd", "provisional"]:
        result = compare_snapshot_to_final(table_name, label)
        if not result.empty:
            accuracy_results[(table_name, label)] = result
            print(table_name, "--", label)
            display(result)

### National annual totals

EIA explicitly warns that early-release data "is inappropriate for aggregation, such as to state or national
totals." Sum each numeric column nationally per snapshot and compare directly to the final total to test that
warning.

In [ ]:
national_totals = []
for table_name, spec in TABLES.items():
    for col in spec["numeric_cols"]:
        final_total = snapshots[(table_name, "final")][col].sum()
        for label in ["incremental_ytd", "provisional"]:
            snap_total = snapshots[(table_name, label)][col].sum()
            national_totals.append(
                {
                    "table": table_name,
                    "column": col,
                    "snapshot": label,
                    "snapshot_total": snap_total,
                    "final_total": final_total,
                    "pct_diff": round((snap_total - final_total) / final_total * 100, 2)
                    if final_total
                    else np.nan,
                }
            )

national_totals = pd.DataFrame(national_totals)
national_totals

In [ ]:
fig = px.bar(
    national_totals,
    x="table",
    y="pct_diff",
    color="snapshot",
    facet_row="column",
    barmode="group",
    title="% difference from final national total, by table/column/snapshot",
)
fig.update_layout(height=250 * national_totals["column"].nunique())
fig

## Section 4: Summary

Side-by-side completeness and accuracy for the three EIA-923 tables, so the tradeoff between an early/incremental
OGE release and waiting for `final` data can be scanned at a glance.

In [ ]:
summary_rows = []
for table_name, spec in TABLES.items():
    completeness = completeness_results[table_name]
    for label in ["incremental_ytd", "provisional"]:
        row = {"table": table_name, "snapshot": label}
        snap_completeness = completeness[completeness["snapshot"] == label]
        row["pct_present"] = round(snap_completeness["pct_present"].mean(), 1)
        row["n_extra_in_snapshot"] = snap_completeness["n_extra_in_snapshot"].iloc[0]

        accuracy = accuracy_results.get((table_name, label))
        if accuracy is not None:
            exact_or_close = accuracy.reindex(["!exact", "+/-1%"]).sum().sum()
            row["pct_rows_within_1pct_of_final"] = round(
                exact_or_close / accuracy.sum().sum() * 100, 1
            )
        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary